# Baseline Model
Logistic Regression baseline on the new real dataset feature matrix.

> **Note:** NaN values appear due to the departmental aggregation join strategy — any dept/role combos not found in the perf dataset get NaN, which we fill with the column median before training.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv('../data/processed/features/attrition_features.csv')
X = df.drop(columns=['Attrition_encoded'])
y = df['Attrition_encoded']

# Fill any NaN (from dept-level agg join misses) with column median
nan_before = X.isnull().sum().sum()
if nan_before > 0:
    X = X.fillna(X.median(numeric_only=True))
    print(f'Filled {nan_before} NaN values with median.')
print(f'Feature matrix: {X.shape} | NaN remaining: {X.isnull().sum().sum()}')
print(f'Target: {y.value_counts().to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = LogisticRegression(max_iter=2000, class_weight='balanced')
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
preds = model.predict(X_test)

print(f'ROC-AUC: {roc_auc_score(y_test, probs):.4f}')
print('Classification Report:')
print(classification_report(y_test, preds))

Feature matrix: (500, 42) | NaN remaining: 0
Target: {0: 445, 1: 55}


ROC-AUC: 0.9867
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98        89
           1       0.79      1.00      0.88        11

    accuracy                           0.97       100
   macro avg       0.89      0.98      0.93       100
weighted avg       0.98      0.97      0.97       100



E:\enterprise_hr_ai\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
